## Bibliotecas / configuração

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados 
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Funções customizadas
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Modelos
import lightgbm as lgb

# Avisos
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')

### Parâmetros globais

In [ ]:
# Parãmetros Globais
TARGET = 'FPD'
RANDOM_STATE = 42
TOP_N_FEATURES = 20

## Carregar Arquivo

In [ ]:
# Carregar dados
abt01_train = pd.read_parquet(PROCESSED_DIR / 'df_train.parquet')
abt01_test = pd.read_parquet(PROCESSED_DIR / 'df_test.parquet')

print(f'✅ Dados de Treino: {abt01_train.shape[0]:,} registros × {abt01_train.shape[1]} features')
print(f'\nDistribuição do Target (FPD):')
print(f"  Treino: {(abt01_train['FPD'].value_counts(normalize=True) * 100).round(2)}")

### Verificação dos dados

In [ ]:
metadados = dataset_info_table(abt01_train)

### Tratamento inicial

In [ ]:
# Carregar features lista
with open(ARTIFACT_DIR / 'features_list.pkl', 'rb') as f:
    features_list = pickle.load(f)

print(f"✅ Total de features: {len(features_list)}")

# Separar X e y
X = abt01_train[features_list].copy()
y = abt01_train[TARGET].copy()

print(f"\n✅ Dataset de treino:")
print(f"   X_train: {X.shape}")
print(f"   y_train: {y.shape}")

## Seleção de Variáveis

### Seleção de Features com Feature Importance (LightGBM)

In [ ]:
# Ajuste automático de desbalanceamento
pos_weight = (y == 0).sum() / (y == 1).sum()

# Modelo LightGBM raso apenas para ranking de features
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=7,
    min_data_in_leaf=80,
    min_gain_to_split=0.02,
    scale_pos_weight=pos_weight,
    random_state=RANDOM_STATE,
    verbosity=-1
)

# Treinamento
lgb_model.fit(X, y)

# Importância das features por ganho (métrica robusta)
feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": lgb_model.booster_.feature_importance(importance_type="gain")
}).sort_values(by="importance", ascending=False)

# Seleção das Top N features
selected_features = feat_imp.head(TOP_N_FEATURES)["feature"].tolist()
selected_features_df = feat_imp.head(TOP_N_FEATURES)

print(f"Número de features selecionadas: {len(selected_features)}")

In [ ]:
# Lista de variáveis selecionadas
print(list(selected_features_df['feature']))

# lista final de colunas para modelagem
cols_modelo = selected_features + [TARGET]

abt_train_fs01 = abt01_train[cols_modelo]
abt_test_fs01  = abt01_test[cols_modelo]

In [ ]:
# Plot das features selecionadas
plt.figure(figsize=(10, len(selected_features_df) * 0.4))
plt.barh(
    selected_features_df['feature'],
    selected_features_df['importance']
)
plt.xlabel("Feature Importance (gain)")
plt.title("Top Features Selecionadas - LightGBM")
plt.tight_layout()
plt.show()

In [ ]:
# Selecionar apenas as features escolhidas para treino e teste
X_train_fs = abt01_train[selected_features]
X_test_fs  = abt01_test[selected_features]

### Correlação

In [ ]:
# Objetivo: exibir valores numéricos dentro do heatmap de correlação
corr = X_train_fs.select_dtypes(include='number').corr()

# Mostrar apenas metade da matriz
mask = np.triu(np.ones_like(corr, dtype = bool))

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,          # MOSTRA os números
    fmt=".2f",           # duas casas decimais
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 9},  # tamanho da fonte
    cbar_kws={"shrink": 0.8}
)

plt.title("Heatmap de Correlação — Features pós Feature Engineering")
plt.tight_layout()
plt.show()

In [ ]:
# Identificar features altamente correlacionadas para remoção
corr = X_train_fs.select_dtypes(include='number').corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Identificar variáveis com correlação acima do limiar (ex: 0.9) para possível remoção
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
to_drop

In [ ]:
# Remover features colineares com decisão explícita
X_train_fs = X_train_fs.drop(columns=to_drop)
X_test_fs  = X_test_fs.drop(columns=to_drop)

## Cardinalidade

In [ ]:
metadados = dataset_info_table(X_train_fs)

In [ ]:
# filtra features com cardinalidade maior que 800
limite_cardinalidade = 800

# Remover features com cardinalidade absurda e alto risco de overfitting
cols_drop = (metadados.loc[metadados['Cardinalidade'] > limite_cardinalidade, 'Feature'].tolist())
cols_drop

In [ ]:
# Aplicar o mesmo drop no conjunto de treino e teste para manter consistência do pipeline
X_train_fs.drop(columns=cols_drop, inplace=True, errors='ignore')
X_test_fs.drop(columns=cols_drop, inplace=True, errors='ignore')

In [ ]:
# Checar se número de linhas bate
print("Treino:")
print("linhas abt01:", len(abt01_train))
print("linhas df_train:", len(X_train_fs))

print("\nTeste:")
print("linhas abt01_test:", len(abt01_test))
print("linhas df_test:", len(X_test_fs))

In [ ]:
# Reanexar target e o controle temporal antes de salvar

# Para treino
abt01_train_final = X_train_fs.copy()
#abt01_train_final['SAFRA'] = abt01_train['SAFRA']
abt01_train_final['FPD'] = abt01_train['FPD']  # reanexa target original

# Para teste
abt01_test_final = X_test_fs.copy()
#abt01_test_final['SAFRA'] = abt01_test['SAFRA']
abt01_test_final['FPD'] = abt01_test['FPD']    # reanexa target original

### Analisando as Variaveis

In [ ]:
# Listar as colunas do dataframe abt01_train_final
lista_colunas = abt01_train_final.columns.tolist()

In [ ]:
# Dividir lista_colunas em blocos mantendo FPD em todas
target = 'FPD'
features = [c for c in lista_colunas if c != target]

tamanho_bloco = 4  # número de variáveis além do target

listas = [
    [target] + features[i:i + tamanho_bloco]
    for i in range(0, len(features), tamanho_bloco)
]

In [ ]:
# objetivo: gerar pairplot para cada lista de variáveis
for i, cols in enumerate(listas, start=1):
    abt_sample = abt01_train_final.sample(n=5000, random_state=42)

    sns.pairplot(
        abt_sample[cols],
        hue='FPD',
        palette='Blues'
    )

    plt.suptitle(f'Pairplot - Lista {i}', y=1.02)
    plt.show()

## Sanity check

In [ ]:
cols_train = set(abt01_train_final.columns)
cols_test = set(abt01_test_final.columns)

print('Só no treino:', cols_train - cols_test)
print('Só no teste:', cols_test - cols_train)

## Salvamento dos Dados Processados

In [ ]:
# Salvar datasets processados e lista de features

print('\n💾 Salvando dados processados...')

# Dataset completo (opcional)
abt01_train_final.to_parquet(PROCESSED_DIR / 'abt01_train_fs.parquet', index=False)
abt01_test_final.to_parquet(PROCESSED_DIR / 'abt01_test_fs.parquet', index=False)
print(f'   ✓ Treino salvo: {PROCESSED_DIR / "abt01_train_fs.parquet"}')
print(f'   ✓ Teste salvo: {PROCESSED_DIR / "abt01_test_fs.parquet"}')

# Salvar lista de features (excluindo a target)
selected_features = [c for c in X_train_fs.columns if c != 'FPD']
with open(ARTIFACT_DIR / 'selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)
print(f'   ✓ Lista de features salva em: {ARTIFACT_DIR / "selected_features.pkl"}')

print(f'\n✅ Todos os dados processados foram salvos')